# Silent Signal — Graph Encoder Check cho ASL Citizen top-200

Notebook 06 đọc graph cache do notebook 05 tạo, dựng Graph-Spatial-Temporal Encoder và classifier 200 lớp, kiểm tra forward/backward, mask invariance, chạy vài bước smoke-training rồi lưu checkpoint/report lên Drive. Đây là kiểm tra kiến trúc, **chưa phải full training nhiều epoch**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-graph-encoder'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
DEVICE = 'auto'  # @param ['auto', 'cuda', 'cpu']
SAMPLE_LIMIT = 64  # @param {type:'integer'}
BATCH_SIZE = 8  # @param {type:'integer'}
TRAIN_STEPS = 5  # @param {type:'integer'}
RUN_SMOKE_TRAINING = True  # @param {type:'boolean'}
OVERWRITE_CHECKPOINT = True  # @param {type:'boolean'}

EXPECTED_CLIPS = 6146
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
SUBSET_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200'
MANIFEST = SUBSET_ROOT / 'manifest.csv'
GRAPH_ROOT = SUBSET_ROOT / 'graph/asl_citizen_coco_wholebody_v1/t64'
GRAPH_PREP_REPORT = SUBSET_ROOT / 'reports/graph_preparation_t64.json'
MODEL_ROOT = SUBSET_ROOT / 'models/graph_encoder_v1'
CHECKPOINT = MODEL_ROOT / 'smoke_checkpoint.pt'
REPORT = SUBSET_ROOT / 'reports/graph_encoder_smoke.json'
PROJECT_ROOT = Path('/content/silent-signal')

## Lấy đúng nhánh và kiểm tra PyTorch/GPU

In [ ]:
import shutil, subprocess, sys
def run(command):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    subprocess.run(command, check=True)
if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', PROJECT_ROOT])
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Chặn chạy nếu notebook 05 chưa hoàn tất

In [ ]:
import json
if not MANIFEST.is_file(): raise FileNotFoundError(MANIFEST)
if not GRAPH_PREP_REPORT.is_file(): raise FileNotFoundError(GRAPH_PREP_REPORT)
graph_report = json.loads(GRAPH_PREP_REPORT.read_text(encoding='utf-8'))
graph_count = sum(1 for _ in GRAPH_ROOT.glob('*/*.npz'))
completed = graph_report.get('prepared', 0) + graph_report.get('resumed', 0)
print('Graph cache:', graph_count, '/', EXPECTED_CLIPS)
print('Graph report:', completed, 'done,', graph_report.get('failed'), 'failed')
if graph_count != EXPECTED_CLIPS or completed != EXPECTED_CLIPS or graph_report.get('failed') != 0:
    raise RuntimeError('Notebook 05 chưa PASS đủ 6146 graph cache.')

## Forward/backward và smoke-training
Checkpoint này chứng minh encoder hoạt động đúng; độ chính xác sau 5 bước không có ý nghĩa đánh giá mô hình.

In [ ]:
commit = subprocess.check_output(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD'], text=True).strip()
command = [sys.executable, '-m', 'silent_signal.cli.check_graph_encoder',
    '--config', PROJECT_ROOT / 'configs/model/asl_citizen_graph_encoder.yaml',
    '--manifest', MANIFEST, '--graph-root', GRAPH_ROOT,
    '--checkpoint', CHECKPOINT, '--report', REPORT,
    '--device', DEVICE, '--limit', str(SAMPLE_LIMIT),
    '--batch-size', str(BATCH_SIZE), '--train-steps', str(TRAIN_STEPS),
    '--project-commit', commit]
if OVERWRITE_CHECKPOINT: command.append('--overwrite')
if RUN_SMOKE_TRAINING: run(command)
else: print('RUN_SMOKE_TRAINING=False; chưa chạy encoder check.')

## Kết luận và artifact trên Drive

In [ ]:
if REPORT.is_file():
    result = json.loads(REPORT.read_text(encoding='utf-8'))
    print(json.dumps(result, ensure_ascii=False, indent=2))
    if result.get('state') == 'passed' and CHECKPOINT.is_file():
        print('PASS: graph encoder forward/backward và checkpoint đều hợp lệ.')
        print('Bước tiếp theo: full training trên train, chọn checkpoint bằng validation.')
else:
    print('Chưa có report:', REPORT)